# CivilComments rubric scoring on Colab GPU

Runs the 30-slice dose-response scoring (6 doses × 5 seeds) using your existing
`data_risk_rubric` + `data_risk_experiments` packages, with Detoxify forced onto the GPU.

**Before you start:** `Runtime → Change runtime type → GPU`. Then copy your two
*edited* package folders (`data_risk_rubric/` and `data_risk_experiments/` — the ones
with the seed-replication loader and the `google/civil_comments` path) into a folder in
your Google Drive, e.g. `MyDrive/data_quality_risk/`.

In [ ]:
# 1. Confirm the GPU is actually attached.
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY')
# If this prints 'CPU ONLY', go to Runtime → Change runtime type → GPU, then rerun.

In [ ]:
# 2. Mount Drive, copy your packages to local disk, and install them.
#    (We copy to /content first — pip install directly off a Drive FUSE path is flaky.)
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/data_quality_risk'   # <-- adjust to your Drive folder

!cp -r "{PROJECT}/data_risk_rubric" /content/
!cp -r "{PROJECT}/data_risk_experiments" /content/

%pip install -q "/content/data_risk_rubric[ml]"
%pip install -q "/content/data_risk_experiments[text]"
%pip install -q detoxify datasets
print('install done')

### Force Detoxify onto the GPU
Your `harm_content_density` proxy does `from detoxify import Detoxify` / `Detoxify('original')`
with no device, which defaults to **CPU**. The cell below patches the constructor so every
instantiation lands on CUDA. Because the import inside the proxy happens at call time, this
patch takes effect without touching your source. (For the committed repo, the cleaner fix is
to edit `safety.py` to pass `device='cuda' if torch.cuda.is_available() else 'cpu'`.)

In [ ]:
# 3. GPU patch for Detoxify.
import detoxify, torch
_DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
_orig_Detoxify = detoxify.Detoxify
def _Detoxify_gpu(*args, **kwargs):
    kwargs.setdefault('device', _DEV)
    return _orig_Detoxify(*args, **kwargs)
detoxify.Detoxify = _Detoxify_gpu
print('Detoxify will run on:', _DEV)

In [ ]:
# 4. Run the full 30-slice scoring. Outputs + HF cache go to Drive so they survive
#    a runtime reset and you don't re-download CivilComments next session.
from pathlib import Path
from data_risk_experiments.config import CIVILCOMMENTS
from data_risk_experiments.scoring import score_one_dataset

OUT_DIR = Path(PROJECT) / 'results' / 'rubric_scores'
CACHE   = str(Path(PROJECT) / 'data_cache')

out = score_one_dataset(
    CIVILCOMMENTS,
    output_dir=OUT_DIR,
    cache_dir=CACHE,
    seed=42,
    verbose=True,
)
print('\nwrote', OUT_DIR / 'civilcomments_rubric_scores.json')
print('n slices:', out['n_slices'])   # expect 30

In [ ]:
# 5. Quick sanity check: does harm_content_density climb with the injected dose?
#    (This is also a preview of the flat CSV we'll build next.)
import json, pandas as pd
data = json.load(open(OUT_DIR / 'civilcomments_rubric_scores.json'))
rows = []
for s in data['slices']:
    injected_p = int(s['name'].split('_p')[1].split('_')[0]) / 1000.0
    seed_idx   = s['name'].split('_s')[-1]
    hcd = s['per_sub_dimension']['safety'].get('harm_content_density')
    rows.append((injected_p, seed_idx, hcd, s['S']))
df = pd.DataFrame(rows, columns=['injected_p','seed','harm_content_density','S'])
print(df.groupby('injected_p')[['harm_content_density','S']].mean())
# harm_content_density should rise monotonically from p=0.0 to p=0.4.